In [169]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC, SVC, NuSVC
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.model_selection import GridSearchCV
from utils import *

In [ ]:
results = {"CV": np.array([]),
           "ACC": np.array([]),
           "PRE": np.array([]),
           "REC": np.array([]),
           "F1": np.array([])}

comments, sentiments = load_data("dataset/reviews.csv")

comments.

# create dictionary mapping from emoji to sentiment
emoji_dict = create_emoji_dict(comments)
for i in range(len(comments)):
    comments[i] = normalize_emoji(comments[i], emoji_dict)
    comments[i] = clean_text(comments[i])
    comments[i] = remove_stopwords(comments[i])

# remove null comments
mask = comments != ""
comments = comments[mask]
sentiments = sentiments[mask]


permutations = np.random.permutation(len(comments))
comments = comments[permutations]
sentiments = sentiments[permutations]

In [171]:
comments[0]

'راه عندي مشكل لبيس جيزي ردوا علينا'

In [172]:
X_train, X_test, y_train, y_test = train_test_split(comments, sentiments, test_size=0.02)

vectorizer = TfidfVectorizer(ngram_range=(1, 2))

X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

In [173]:
# from sklearn.utils.class_weight import compute_class_weight

# weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
# classes = np.unique(y_test)
# weight = {classes[i]:weights[i] for i in range(3)}

# params = {
#     "kernel": ["linear", "rbf", "poly", "sigmoid"],
#     "cache_size": [200, 500, 1000],
#     "class_weight": ["balanced", None],
#     "verbose": [1, 2, 5],
#     "max_iter": [1000, 2500, 5000],
#     "random_state": [42]
# }


# model = SVC(random_state=42)

# search = GridSearchCV(model, params, cv=5)
# search.fit(X_train_vector, y_train)

# # model.fit(X_train_vector, y_train)
# # y_pred = model.predict(X_test_vector)

# # accuracy = accuracy_score(y_test, y_pred)
# # print("Accuracy: ", accuracy)

Best params:  {'alpha': 0.0001, 'class_weight': 'balanced', 'fit_intercept': True, 'loss': 'hinge', 'penalty': 'l2'}  
Best CV score: 0.6505859375  
Test score:    0.5904761904761905  

Best params:  {'alpha': 0.0001, 'class_weight': 'balanced', 'fit_intercept': True, 'l1_ratio': 0.1, 'learning_rate': 'optimal', 'loss': 'log_loss',  'penalty': 'l2', 'shuffle': False, 'tol': 1e-05}  
Best CV score: 0.7443359375  
Test score:    0.7047619047619048  

In [174]:
from sklearn.linear_model import SGDClassifier

param_grid = [
    {  
        "loss":          ["hing", "squared_hinge"],
        "penalty":       ["l2", "l1", "elasticnet"],        
        "alpha":         [0.0001, 0.001, 0.01, 0.1, 0.3, 0.9],
        "fit_intercept": [True, False],
        "class_weight":  ["balanced", None],
        "l1_ratio":      [0.01, 0.02, 0.04, 0.08, 0.1, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18],
        "shuffle":       [False],
        "tol":           [1e-3, 1e-4, 1e-5],
        "learning_rate": ["optimal", "adaptive"],
        "et0":           [0.9, 0.3, 0.1, 0.01, 0.001, 0.0001]
    },
]

model = SGDClassifier(
    max_iter=100000,
    alpha=0.0001,
    class_weight="balanced",
    fit_intercept=True,
    l1_ratio=0.1,
    learning_rate="optimal",
    loss="log_loss",
    penalty="l2",
    shuffle=False,
    tol=1e-05)

model.fit(X_train_vector, y_train)
y_pred = model.predict(X_test_vector)

print(accuracy_score(y_test, y_pred))

0.7619047619047619


In [175]:
param_grid = [
    {  
        "loss":          ["hinge", "squared_hinge"], 
        "penalty":       ["l2", "l1", "elasticnet"],        
        "alpha":         [0.0001, 0.001, 0.01, 0.1, 0.3, 0.9],
        "fit_intercept": [True, False],
        "class_weight":  ["balanced", None],
        "l1_ratio":      [0.01, 0.05, 0.1, 0.2, 0.5, 0.8], 
        "shuffle":       [True], 
        "tol":           [1e-3, 1e-4, 1e-5],
        "learning_rate": ["optimal", "adaptive"],
        "eta0":          [0.1, 0.01, 0.001] 
    },
]

clf = GridSearchCV(
    SGDClassifier(max_iter=100000),
    param_grid,
    cv=5,
    scoring="accuracy",
)
clf.fit(X_train_vector, y_train)

print("Best params: ", clf.best_params_)
print("Best CV score:", clf.best_score_)
print("Test score:   ", clf.best_estimator_.score(X_test_vector, y_test))

KeyboardInterrupt: 